In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import pandas as pd
import os

# ============================================
# 1. Custom Dataset for DataFrame-based inputs
# ============================================

class ImageDFDataset(Dataset):
    def __init__(self, df, label_to_idx, transform=None):
        self.df = df.reset_index(drop=True)
        self.label_to_idx = label_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = '../data' + self.df.loc[idx, "image_path"]
        label_str = self.df.loc[idx, "label"]
        label = self.label_to_idx[label_str]

        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label


# ============================================
# 2. Build label mapping
# ============================================

def build_label_mapping(df_train):
    classes = sorted(df_train["label"].unique())
    label_to_idx = {c: i for i, c in enumerate(classes)}
    idx_to_label = {i: c for c, i in label_to_idx.items()}
    return label_to_idx, idx_to_label


# ============================================
# 3. Transforms
# ============================================

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(128, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.1),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.5),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(144),
    transforms.CenterCrop(128),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


# ============================================
# 4. Create datasets & loaders from dataframe
# ============================================

def make_dataloaders(df_train, df_val, batch_size=32):

    label_to_idx, idx_to_label = build_label_mapping(df_train)
    num_classes = len(label_to_idx)

    train_dataset = ImageDFDataset(df_train, label_to_idx, transform=train_transform)
    val_dataset   = ImageDFDataset(df_val, label_to_idx, transform=val_transform)

    # ---- Balanced sampler (important for ~20 images/class) ----
    class_counts = df_train["label"].value_counts().sort_index()
    class_weights = 1.0 / torch.tensor(class_counts.tolist(), dtype=torch.float)

    sample_weights = [
        class_weights[label_to_idx[label]]
        for label in df_train["label"]
    ]

    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, num_classes, label_to_idx, idx_to_label


# ============================================
# 5. Build ResNet-18 (train from scratch)
# ============================================

def build_resnet18(num_classes):
    model = models.resnet18(weights=None)   # NOT pretrained

    model.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(512, num_classes)
    )

    return model


# ============================================
# 6. Training loop (clean version)
# ============================================

def train_model(model, train_loader, val_loader, epochs=100, lr=1e-3):

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(epochs):
        model.train()
        running_loss = 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        val_loss = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        print(f"Epoch {epoch+1}/{epochs} "
              f"| Train Loss: {running_loss/len(train_loader):.4f} "
              f"| Val Loss: {val_loss:.4f}")


# ============================================
# 7. Validation
# ============================================

@torch.inference_mode()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item()

    return total_loss / len(loader)




In [2]:
from sklearn.model_selection import train_test_split
# ============================================
# 8. Example usage
# ============================================
df = pd.read_csv("../data/train_images.csv")
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

train_loader, val_loader, num_classes, label_to_idx, idx_to_label = \
    make_dataloaders(train_df, val_df)

model = build_resnet18(200)

In [3]:
train_model(model, train_loader, val_loader, epochs=120, lr=1e-3)

Epoch 1/120 | Train Loss: 5.5573 | Val Loss: 5.8138
Epoch 2/120 | Train Loss: 5.3296 | Val Loss: 5.2428
Epoch 3/120 | Train Loss: 5.2556 | Val Loss: 5.2220
Epoch 4/120 | Train Loss: 5.2394 | Val Loss: 5.4139
Epoch 5/120 | Train Loss: 5.1837 | Val Loss: 5.1863
Epoch 6/120 | Train Loss: 5.1489 | Val Loss: 5.1996
Epoch 7/120 | Train Loss: 5.1324 | Val Loss: 5.1362
Epoch 8/120 | Train Loss: 5.1233 | Val Loss: 5.2065
Epoch 9/120 | Train Loss: 5.0963 | Val Loss: 5.1553
Epoch 10/120 | Train Loss: 5.0364 | Val Loss: 5.0737
Epoch 11/120 | Train Loss: 4.9987 | Val Loss: 5.0697
Epoch 12/120 | Train Loss: 4.9191 | Val Loss: 5.0203
Epoch 13/120 | Train Loss: 4.8886 | Val Loss: 4.8881
Epoch 14/120 | Train Loss: 4.8148 | Val Loss: 4.8382
Epoch 15/120 | Train Loss: 4.7719 | Val Loss: 4.7647
Epoch 16/120 | Train Loss: 4.7022 | Val Loss: 4.8157
Epoch 17/120 | Train Loss: 4.6426 | Val Loss: 4.8125
Epoch 18/120 | Train Loss: 4.6350 | Val Loss: 4.7256
Epoch 19/120 | Train Loss: 4.5716 | Val Loss: 4.6381
Ep

In [4]:
test_df = pd.read_csv("../data/test_images_path.csv")
test_dataset   = ImageDFDataset(test_df, label_to_idx, transform=val_transform)
test_loader   = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [5]:
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [6]:
all_ids = test_df["id"].tolist()
all_preds = []

In [7]:
with torch.no_grad():
    for inputs, _ in test_loader:
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.numpy())

In [8]:
predicted_labels = [idx_to_label[i] for i in all_preds]

In [9]:
output_df = pd.DataFrame({
    "id": all_ids,
    "label": predicted_labels
})

output_df.to_csv("test_predictions.csv", index=False)
print("Saved test_predictions.csv!")

Saved test_predictions.csv!


In [11]:
from sklearn.metrics import accuracy_score
val_preds = []
val_labels = []
model.eval()

with torch.no_grad():
    for inputs, labels in val_loader:
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)

        val_preds.extend(predicted.numpy())
        val_labels.extend(labels.numpy())

accuracy = accuracy_score(val_labels, val_preds)

In [12]:
accuracy

0.2455470737913486